# Criteo Uplift Analysis

This notebook analyzes a large randomized advertising experiment from Criteo (about 14 million users, roughly 85% treated). It covers a data audit, A/B test analysis (average treatment effect, confidence intervals, power, intent-to-treat vs. exposure effects), uplift modeling evaluated with Qini curves, and a targeting decision that estimates the value of showing ads only to the users most likely to respond.

## 0. Setup

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import duckdb
import sklearn
import lightgbm as lgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SEED = 42
np.random.seed(SEED)

print("pandas   ", pd.__version__)
print("numpy    ", np.__version__)
print("duckdb   ", duckdb.__version__)
print("sklearn  ", sklearn.__version__)
print("lightgbm ", lgb.__version__)

pandas    2.3.3
numpy     2.1.3
duckdb    1.3.2
sklearn   1.6.1
lightgbm  4.6.0


### 0.1 Locate input files

In [2]:
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{path}  ({size_mb:,.1f} MB)")

/kaggle/input/datasets/guilinzhao/criteo-uplift-parquet/criteo_uplift.parquet  (104.5 MB)
/kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv  (3,097.6 MB)


### 0.2 First look at the raw file

In [3]:
csv_files = sorted(glob.glob("/kaggle/input/**/*.csv*", recursive=True))
print(csv_files)

RAW_PATH = csv_files[0]
print("Using:", RAW_PATH)

sample = pd.read_csv(RAW_PATH, nrows=5)
print(sample.shape)
print(sample.dtypes)
sample

['/kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv']
Using: /kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv
(5, 16)
f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.6164,10.0597,8.9764,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
1,12.6164,10.0597,9.0027,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
2,12.6164,10.0597,8.9648,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
3,12.6164,10.0597,9.0028,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
4,12.6164,10.0597,9.0380,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0


### 0.3 Convert CSV to Parquet (run once)

The raw CSV is about 3.1 GB. It is converted once to a compressed Parquet file with compact types (float32 features, int8 labels) and saved as a Kaggle dataset. Later runs set `RUN_DATA_PREP = False` and read the saved file directly.

In [4]:
import time

RUN_DATA_PREP = False  # Set to False after the Parquet file is saved as a Kaggle dataset

PARQUET_NAME = "criteo_uplift.parquet"
WORK_PATH = f"/kaggle/working/{PARQUET_NAME}"

FEATURE_COLS = [f"f{i}" for i in range(12)]
LABEL_COLS = ["treatment", "conversion", "visit", "exposure"]

if RUN_DATA_PREP:
    start = time.time()
    con = duckdb.connect()

    select_sql = ",\n        ".join(
        [f"CAST({c} AS FLOAT) AS {c}" for c in FEATURE_COLS]
        + [f"CAST({c} AS TINYINT) AS {c}" for c in LABEL_COLS]
    )

    con.execute(f"""
        COPY (
            SELECT
        {select_sql}
            FROM read_csv('{RAW_PATH}', header = true)
        ) TO '{WORK_PATH}' (FORMAT parquet, COMPRESSION zstd)
    """)

    raw_rows = con.execute(
        f"SELECT COUNT(*) FROM read_csv('{RAW_PATH}', header = true)"
    ).fetchall()[0][0]
    pq_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{WORK_PATH}')"
    ).fetchall()[0][0]
    con.close()

    print(f"Raw CSV rows:     {raw_rows:,}")
    print(f"Parquet rows:     {pq_rows:,}")
    print(f"Row counts match: {raw_rows == pq_rows}")
    print(f"Parquet size:     {os.path.getsize(WORK_PATH) / 1024**2:,.1f} MB")
    print(f"Elapsed:          {time.time() - start:,.1f} s")
else:
    print("Skipping data prep; the saved Parquet will be loaded below.")

Skipping data prep; the saved Parquet will be loaded below.


### 0.4 Load the prepared data

In [5]:
if RUN_DATA_PREP:
    DATA_PATH = WORK_PATH
else:
    found = glob.glob(f"/kaggle/input/**/{PARQUET_NAME}", recursive=True)
    assert found, "Prepared Parquet not found. Attach the prepared dataset as an input."
    DATA_PATH = found[0]

df = pd.read_parquet(DATA_PATH)

print("Loaded from:", DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
print(df.dtypes)
df.head()

Loaded from: /kaggle/input/datasets/guilinzhao/criteo-uplift-parquet/criteo_uplift.parquet
Shape: 13,979,592 rows x 16 columns
Memory: 693.3 MB
f0            float32
f1            float32
f2            float32
f3            float32
f4            float32
f5            float32
f6            float32
f7            float32
f8            float32
f9            float32
f10           float32
f11           float32
treatment        int8
conversion       int8
visit            int8
exposure         int8
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.6164,10.0597,8.9764,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
1,12.6164,10.0597,9.0027,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
2,12.6164,10.0597,8.9648,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
3,12.6164,10.0597,9.0028,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
4,12.6164,10.0597,9.0380,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0


## 1. Data audit

Before estimating any effect, the data is checked for completeness, valid label values, consistency between labels, and whether treatment assignment looks random.

### 1.1 Missing values, label values, and duplicates

In [6]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nDistinct values in label columns:")
for c in LABEL_COLS:
    print(f"  {c:<11}", sorted(df[c].unique().tolist()))

n_dup = df.duplicated().sum()
n_dup_feat = df.duplicated(subset=FEATURE_COLS).sum()
print(f"\nExact duplicate rows (all 16 columns): {n_dup:,} ({n_dup / len(df):.2%})")
print(f"Rows with duplicated features only:    {n_dup_feat:,} ({n_dup_feat / len(df):.2%})")

Missing values per column:
None

Distinct values in label columns:
  treatment   [0, 1]
  conversion  [0, 1]
  visit       [0, 1]
  exposure    [0, 1]

Exact duplicate rows (all 16 columns): 1,655,747 (11.84%)
Rows with duplicated features only:    2,120,151 (15.17%)


### 1.2 Group sizes and outcome rates

In [7]:
def group_summary(data):
    return pd.Series({
        "users": len(data),
        "visit_rate": data["visit"].mean(),
        "conversion_rate": data["conversion"].mean(),
        "exposure_rate": data["exposure"].mean(),
    })

summary = df.groupby("treatment").apply(group_summary)
summary.index = summary.index.map({0: "control", 1: "treatment"})
summary.loc["overall"] = group_summary(df)
summary["share"] = summary["users"] / len(df)

summary.style.format({
    "users": "{:,.0f}",
    "share": "{:.2%}",
    "visit_rate": "{:.3%}",
    "conversion_rate": "{:.4%}",
    "exposure_rate": "{:.3%}",
})

,users,visit_rate,conversion_rate,exposure_rate,share
treatment,,,,,
control,"2,096,937",3.820%,0.1938%,0.000%,15.00%
treatment,"11,882,655",4.854%,0.3089%,3.604%,85.00%
overall,"13,979,592",4.699%,0.2917%,3.063%,100.00%


### 1.3 Logical consistency between labels

Expected relationships: control users should never be exposed to the ad, and a conversion should normally follow a visit.

In [8]:
print("Treatment x exposure:")
print(pd.crosstab(df["treatment"], df["exposure"], margins=True), "\n")

print("Visit x conversion:")
print(pd.crosstab(df["visit"], df["conversion"], margins=True), "\n")

checks = {
    "Control users with exposure = 1": ((df["treatment"] == 0) & (df["exposure"] == 1)).sum(),
    "Conversions without a visit": ((df["conversion"] == 1) & (df["visit"] == 0)).sum(),
}
for name, n in checks.items():
    print(f"{name:<34} {n:,}")

treated = df[df["treatment"] == 1]
print(f"\nShare of treated users actually exposed: {treated['exposure'].mean():.2%}")

Treatment x exposure:
exposure          0       1       All
treatment                            
0           2096937       0   2096937
1          11454443  428212  11882655
All        13551380  428212  13979592 

Visit x conversion:
conversion         0      1       All
visit                                
0           13322663      0  13322663
1             616155  40774    656929
All         13938818  40774  13979592 

Control users with exposure = 1    0
Conversions without a visit        0

Share of treated users actually exposed: 3.60%


### 1.4 Randomization check: covariate balance

If assignment is random, feature distributions should be nearly identical across groups. With 14 million rows, even tiny differences are statistically significant, so balance is judged by the standardized mean difference (SMD) rather than p-values. An absolute SMD below 0.1 is a common threshold for good balance.

In [9]:
is_treated = df["treatment"] == 1
rows = []
for c in FEATURE_COLS:
    x = df[c].astype("float64")
    m1, m0 = x[is_treated].mean(), x[~is_treated].mean()
    v1, v0 = x[is_treated].var(), x[~is_treated].var()
    rows.append({
        "feature": c,
        "mean_treatment": m1,
        "mean_control": m0,
        "smd": (m1 - m0) / np.sqrt((v1 + v0) / 2),
    })

balance = pd.DataFrame(rows).set_index("feature")
balance["abs_smd"] = balance["smd"].abs()
print(f"Max |SMD|: {balance['abs_smd'].max():.4f} ({balance['abs_smd'].idxmax()})")
print(f"Features with |SMD| >= 0.1: {(balance['abs_smd'] >= 0.1).sum()}")
balance.sort_values("abs_smd", ascending=False)

Max |SMD|: 0.0488 (f3)
Features with |SMD| >= 0.1: 0


,mean_treatment,mean_control,smd,abs_smd
feature,,,,
f3,4.1694,4.2328,-0.0488,0.0488
f6,-4.1828,-3.9999,-0.0404,0.0404
f5,4.0266,4.0393,-0.0306,0.0306
f9,16.0526,15.8863,0.0240,0.0240
f1,10.0703,10.0679,0.0240,0.0240
f8,3.9334,3.9347,-0.0224,0.0224
f7,5.1056,5.0803,0.0213,0.0213
f10,5.3337,5.3319,0.0106,0.0106
f4,10.3392,10.3365,0.0080,0.0080


### 1.5 Randomization check: can features predict treatment?

A stronger test: train a classifier to predict treatment from the 12 features. Under true randomization, it should do no better than chance (ROC-AUC close to 0.5). A 1 million-row random sample is enough for this check.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sample = df.sample(n=1_000_000, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    sample[FEATURE_COLS], sample["treatment"],
    test_size=0.3, stratify=sample["treatment"], random_state=SEED,
)

clf = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.1, num_leaves=31,
    random_state=SEED, verbose=-1,
)
clf.fit(X_tr, y_tr)
auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
print(f"ROC-AUC for predicting treatment from features: {auc:.4f}")

ROC-AUC for predicting treatment from features: 0.5126


### 1.6 Permutation baseline for the treatment classifier

To judge whether an AUC of about 0.51 is meaningful, the same classifier is trained on randomly shuffled treatment labels. The spread of these AUCs shows what pure chance looks like at this sample size.

In [11]:
rng = np.random.default_rng(SEED)
perm_aucs = []
for i in range(3):
    y_tr_perm = rng.permutation(y_tr.values)
    y_te_perm = rng.permutation(y_te.values)
    clf_perm = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.1, num_leaves=31,
        random_state=SEED, verbose=-1,
    )
    clf_perm.fit(X_tr, y_tr_perm)
    perm_aucs.append(roc_auc_score(y_te_perm, clf_perm.predict_proba(X_te)[:, 1]))

print(f"Real-label AUC:      {auc:.4f}")
print("Shuffled-label AUCs:", [round(float(a), 4) for a in perm_aucs])

importance = pd.Series(
    clf.booster_.feature_importance(importance_type="gain"), index=FEATURE_COLS
)
print("\nFeatures that best separate treatment from control (gain share):")
print((importance / importance.sum()).sort_values(ascending=False).round(3))

Real-label AUC:      0.5126
Shuffled-label AUCs: [0.4977, 0.5009, 0.4993]

Features that best separate treatment from control (gain share):
f0    0.1670
f2    0.1610
f6    0.1490
f8    0.1480
f9    0.1040
f10   0.0740
f7    0.0730
f3    0.0630
f4    0.0290
f5    0.0150
f11   0.0110
f1    0.0080
dtype: float64


### 1.7 Audit summary

- **Complete and valid.** There are no missing values, and all four labels are binary. About 11.8% of rows are exact duplicates and 15.2% share identical feature values. Because the features are anonymized projections, different users can share identical values, so all rows are kept.
- **Group split.** 85.00% of users are in treatment and 15.00% in control, within 2 users of an exact 85/15 split. The dataset was sub-sampled by its publisher, so a sample ratio mismatch test is not informative here.
- **Labels are consistent.** No control user was exposed to the ad, and every conversion has a visit. Only 3.60% of treated users were actually exposed, so a treatment-vs-control comparison measures the effect of being assigned to treatment (intent-to-treat), not the effect of seeing the ad.
- **Randomization holds, with a small imbalance.** All absolute SMDs are below 0.1 (maximum 0.049 for `f3`), and a classifier predicts treatment with ROC-AUC 0.513. With shuffled labels the AUC stays between 0.498 and 0.501, so the imbalance is small but larger than chance. It is spread across several features rather than concentrated in one. The publisher states that the data was assembled from several incrementality tests, and pooling tests with different populations is a plausible source. Section 2 checks whether this matters by comparing the raw estimate with a regression-adjusted one.
- **Absolute levels are not Criteo's real results.** The publisher sub-sampled the data non-uniformly for privacy, so the original incrementality level cannot be recovered. Effect sizes and value estimates in this notebook describe this dataset, not Criteo's actual campaigns; comparisons between methods within the dataset remain meaningful.